# Gold dataset offline audit — 2026-10-15
审查范围：data/dev.jsonl、data/test.jsonl 与静态 Canvas CASES。
自动检查结构、引用与副本一致性；语义审查见同目录 Markdown。
评分演示是构造的复现，不是模型实测，也不代表人工批准。
从项目根目录或 reviews 目录运行，使用项目 .venv Python 和本机 Node；不访问网络。


In [1]:

import copy
import hashlib
import json
import subprocess
import sys
from collections import Counter
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "assignment.md").exists():
    ROOT = ROOT.parent
assert (ROOT / "assignment.md").exists(), "Run from the repository or reviews directory"
sys.path.insert(0, str(ROOT / "src"))
from grounding_verifier.dataset import load_cases, validate_splits, dataset_hash
from grounding_verifier.evaluation import output_hash, review_template, score

inputs = ["data/dev.jsonl", "data/test.jsonl", "reviews/gold-dataset-review.canvas.tsx"]
before = {p: hashlib.sha256((ROOT / p).read_bytes()).hexdigest() for p in inputs}
cases = load_cases(ROOT / "data/dev.jsonl") + load_cases(ROOT / "data/test.jsonl")
splits = validate_splits(ROOT / "data")

# Parse only the local, static CASES array, not imports, JSX, or component code.
js = r"""
const fs = require('node:fs');
const vm = require('node:vm');
const text = fs.readFileSync(process.argv[1], 'utf8');
const match = text.match(/const CASES: CaseRow\[\] = (\[[\s\S]*?\n\]);/);
if (!match) throw new Error('Static CASES literal was not found');
const cases = vm.runInNewContext('(' + match[1] + ')', Object.create(null), {timeout: 1000});
process.stdout.write(JSON.stringify(cases));
"""
completed = subprocess.run(
    ["node", "-e", js, str(ROOT / inputs[2])],
    capture_output=True, text=True, encoding="utf-8", check=True,
)
canvas = json.loads(completed.stdout)
by_id = {c.id: c for c in cases}
assert len(canvas) == len(cases) == len({c["id"] for c in canvas})
assert set(by_id) == {c["id"] for c in canvas}
compared = 0
for item in canvas:
    original = by_id[item["id"]]
    assert item["split"] == original.id.split("-")[0]
    assert item["category"] == original.category
    assert item["passages"] == original.context_passages
    assert item["answer"] == original.generated_answer
    expected_claims = [
        {
            "id": g.id, "claim_text": g.claim_text, "verdict": g.verdict.value,
            "citations": " · ".join(f'[{c.index}] "{c.quote}"' for c in g.citations) or "[]",
        }
        for g in original.gold_claims
    ]
    assert item["claims"] == expected_claims, original.id
    compared += len(expected_claims)

quotes = valid_quotes = whole_quotes = 0
labels = Counter()
reasons = Counter()
annotations = Counter()
for case in cases:
    for claim in case.gold_claims:
        labels[claim.verdict.value] += 1
        reasons[claim.reasoning] += 1
        annotations[claim.annotation_reason] += 1
        assert bool(claim.citations) == (claim.verdict.value != "UNSUPPORTED")
        for citation in claim.citations:
            quotes += 1
            assert type(citation.index) is int
            assert 0 <= citation.index < len(case.context_passages)
            passage = case.context_passages[citation.index]
            assert citation.quote.strip() and citation.quote in passage
            valid_quotes += 1
            whole_quotes += citation.quote == passage

def duplicates(values):
    return {k: n for k, n in Counter(values).items() if n > 1}

duplicate_contexts = duplicates(
    json.dumps(c.context_passages, ensure_ascii=False) for c in cases
)
duplicate_answers = duplicates(c.generated_answer for c in cases)
assert not duplicate_contexts and not duplicate_answers

# Deliberately synthetic scoring reproduction; not a real model result or a
# completed human review. Even a generous same-gold-ID mapping penalizes a
# semantically correct finer split because the second use becomes an FP.
case = by_id["test-05"]
original_output = {
    "claims": [g.model_dump(mode="json", exclude={"id", "annotation_reason"})
               for g in case.gold_claims]
}
split_output = copy.deepcopy(original_output)
first = copy.deepcopy(split_output["claims"][0])
first["claim_text"] = "GET /status returns HTTP 200."
second = copy.deepcopy(first)
second["claim_text"] = "GET /status returns a JSON body."
split_output["claims"] = [first, second, split_output["claims"][1]]

def synthetic_score(output, mapping):
    record = {"case_id": case.id, "status": "ok", "output": output, "trace": {}}
    run = {"dataset_hash": dataset_hash([case]), "method": "synthetic-audit-only",
           "mode": "e2e", "records": [record]}
    review = review_template(run, {case.id: case})
    review[0].update(status="approved", reviewer="SYNTHETIC REPRODUCTION, NOT HUMAN REVIEW")
    for unit, gold_id in zip(review[0]["units"], mapping, strict=True):
        unit.update(gold_id=gold_id, evidence_sufficient=True,
                    note="Hypothetical mapping to reproduce granularity sensitivity")
    result = score(run, [case], review)
    return {k: result[k] for k in ("f1", "precision", "recall", "atomicity")}

scoring_demo = {
    "kind": "synthetic reproduction; not measured model performance",
    "original_grouping": synthetic_score(original_output, ["g1", "g2"]),
    "finer_correct_split": synthetic_score(split_output, ["g1", "g1", "g2"]),
}
assert scoring_demo["original_grouping"]["f1"] == 1.0
assert scoring_demo["finer_correct_split"]["f1"] == 0.8

manifest = json.loads((ROOT / "data/manifest.json").read_text(encoding="utf-8"))
after = {p: hashlib.sha256((ROOT / p).read_bytes()).hexdigest() for p in inputs}
assert before == after
summary = {
    "scope": inputs, "input_sha256": before, "inputs_unchanged": before == after,
    "schema_and_splits": splits,
    "canvas_case_count": len(canvas), "canvas_matching_claim_count": compared,
    "claim_count": sum(labels.values()), "claim_labels": dict(labels),
    "citation_count": quotes, "valid_verbatim_citations": valid_quotes,
    "whole_passage_citations": whole_quotes,
    "reasoning_templates": dict(reasons), "annotation_reason_templates": dict(annotations),
    "duplicate_contexts": duplicate_contexts, "duplicate_answers": duplicate_answers,
    "passage_count_per_case": dict(Counter(len(c.context_passages) for c in cases)),
    "passage_character_range": [min(len(p) for c in cases for p in c.context_passages),
                                max(len(p) for c in cases for p in c.context_passages)],
    "manifest_status": manifest,
    "test05_synthetic_scoring_demo": scoring_demo,
    "limits": ["Semantic entailment and claim completeness were reviewed manually, not proved by assertions.",
               "No external product facts, paid model calls, or browser rendering were used.",
               "Exact duplicate checks do not prove absence of semantic/template overlap between splits.",
               "No human approval or freeze status was changed."]
}
(ROOT / "reviews/gold-dataset-audit-2026-10-15.checks.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
print(json.dumps(summary, ensure_ascii=False, indent=2))


{
  "scope": [
    "data/dev.jsonl",
    "data/test.jsonl",
    "reviews/gold-dataset-review.canvas.tsx"
  ],
  "input_sha256": {
    "data/dev.jsonl": "ec9c3a3901098a3ad629362b2bab44a5c01c3f5a93981af21aab64838d06d029",
    "data/test.jsonl": "d9a6d43301af6dd8cdb48b4906f0cb024a883af9a4e2b712dee7f9b58d3505fc",
    "reviews/gold-dataset-review.canvas.tsx": "64f4f404dfdf6fdb76fd260940e1ec054e8793a33f5f71403d41d9aae5a9e176"
  },
  "inputs_unchanged": true,
  "schema_and_splits": {
    "dev": {
      "cases": 8,
      "categories": {
        "supported": 2,
        "distortion": 2,
        "extrinsic": 2,
        "multi_passage": 2
      },
      "hash": "529a0ef6c38b97a5dc69b439dc61eabada95b0dcbb8522ad40afd0728f915946",
      "labels": {
        "SUPPORTED": 8,
        "CONTRADICTED": 2,
        "UNSUPPORTED": 2
      }
    },
    "test": {
      "cases": 20,
      "categories": {
        "supported": 5,
        "distortion": 5,
        "extrinsic": 5,
        "multi_passage": 5
      },
 